# Hands-on 1 — Training RQ-KMeans and Inspecting the Semantic ID Structure

## Goal

Derive **semantic IDs** for every item by running residual K-means on item content embeddings, then inspect the resulting token structure:

- How many levels does each semantic ID have?
- How many codes exist per level?
- How is codebook usage distributed across codes?
- How many items collide on the same code tuple, and how is that resolved?

By the end of this notebook we will have a trained tokenizer, a table of semantic IDs, and a set of diagnostics (codebook utilization, entropy, collisions) that tell us whether the tokenizer learned a healthy, well-spread representation.

## Where does RQ-KMeans live in Cornac?

There's no standalone `RQKMeans` recommender class in Cornac. Residual K-means is one of two tokenizers built **into** `cornac.models.TIGER` (`tokenizer="rkmeans"`). TIGER's `fit()` always chains tokenizer training with a full T5-style seq2seq generative-model training stage on top of it, which we don't want yet.

So instead of instantiating `TIGER`, this notebook pulls out just the tokenizer logic. It imports Cornac's own `_kmeans` routine (k-means++ seeding + Lloyd iterations) and re-implements TIGER's `_fit_rkmeans` (level-by-level residual k-means) and `_build_semantic_ids` (collision-disambiguation token) exactly as they appear in `cornac/models/tiger/recom_tiger.py`.

> **Copyright note:** the `RQKMeans` class below mirrors (does not import) the private `TIGER._fit_rkmeans` / `TIGER._build_semantic_ids` methods from [PreferredAI/cornac's `cornac/models/tiger/recom_tiger.py`](https://github.com/PreferredAI/cornac) (Apache 2.0).

### Requirements

This notebook needs `cornac`, `torch`, and `sentence-transformers` installed in your environment.

In [ ]:
from collections import defaultdict

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from cornac.datasets import amazon_review
from cornac.models.tiger.tiger import _kmeans

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {DEVICE}")

# Check Cornac's datasets if you want to swap for another / smaller Amazon
# category (e.g. "office_products")
CATEGORY = "beauty"

## The `RQKMeans` class

`RQKMeans` is a standalone residual-quantized K-means tokenizer. It mirrors Cornac's `TIGER._fit_rkmeans` + `TIGER._build_semantic_ids`.

**How it works:** level-by-level k-means over item content embeddings. At each level we fit k-means on the current residual, assign each item to its nearest centroid, then subtract that centroid before moving to the next level. A final disambiguation level is appended so items that land on an identical code tuple still get a unique semantic ID.

**Parameters**

| Parameter | Default | Description |
|---|---|---|
| `num_levels` | `3` | Number of residual codebooks (semantic-ID levels, before the collision-disambiguation level) |
| `codebook_size` | `256` | Number of clusters (codes) per level |
| `n_iters` | `10` | Lloyd iterations per level's k-means fit (passed to Cornac's `_kmeans`) |
| `device` | `"cpu"` | Torch device to run on |
| `seed` | `None` | Optional random seed for reproducibility |

In [ ]:
class RQKMeans:
    """Residual-quantized K-means semantic-ID tokenizer.

    Level-by-level k-means over item content embeddings: fit k-means on the
    current residual, assign each item to its nearest centroid, subtract
    that centroid, repeat for the next level. A final disambiguation level
    is appended so items landing on an identical code tuple still get a
    unique semantic ID.

    Parameters
    ----------
    num_levels: int, default 3
        Number of residual codebooks (semantic-ID levels, before the
        collision-disambiguation level).
    codebook_size: int, default 256
        Number of clusters (codes) per level.
    n_iters: int, default 10
        Lloyd iterations per level's k-means fit (passed to Cornac's
        `_kmeans`).
    device: str, default "cpu"
    seed: int, optional
    """

    def __init__(self, num_levels=3, codebook_size=256, n_iters=10, device="cpu", seed=None):
        self.num_levels = num_levels
        self.codebook_size = codebook_size
        self.n_iters = n_iters
        self.device = device
        self.seed = seed

    def fit(self, features):
        """features: (N, D) array-like of item content embeddings, one row
        per item. Populates self.sid_table, self.level_sizes, self.centroids,
        self.sid_to_item. Returns self."""
        if self.seed is not None:
            torch.manual_seed(self.seed)

        feats_t = torch.as_tensor(np.asarray(features, dtype="float32"), device=self.device)

        # ---- level-by-level residual k-means ----
        self.centroids = []  # list of (codebook_size, dim) numpy arrays, one per level
        codes = []
        r = feats_t
        for level in range(self.num_levels):
            centroids = _kmeans(r, self.codebook_size, n_iters=self.n_iters)
            ids = torch.cdist(r, centroids).argmin(dim=1)
            r = r - centroids[ids]  # residual fed into the next level
            self.centroids.append(centroids.cpu().numpy())
            codes.append(ids.cpu().numpy())
        codes = np.stack(codes, axis=1).astype("int64")  # (N, num_levels)

        # ---- collision-disambiguation level ----
        counters = defaultdict(int)
        dedup = np.zeros(len(codes), dtype="int64")
        for i, row in enumerate(map(tuple, codes)):
            dedup[i] = counters[row]
            counters[row] += 1

        self.sid_table = np.concatenate([codes, dedup[:, None]], axis=1)  # (N, num_levels + 1)
        self.level_sizes = [self.codebook_size] * self.num_levels + [int(dedup.max()) + 1]
        self.sid_to_item = {tuple(int(v) for v in row): i for i, row in enumerate(self.sid_table)}

        n_collisions = int((dedup > 0).sum())
        print(
            f"Semantic IDs assigned: {len(self.sid_table)} items, "
            f"{n_collisions} collisions, dedup level size {self.level_sizes[-1]}"
        )
        return self

## Step 1 — Load item text and embed with Sentence-T5

RQ-KMeans tokenizes **content embeddings**, not ratings or clicks. We load the raw item text for the chosen Amazon category and encode it with a Sentence-T5 model to get one dense vector per item.

In [ ]:
print(f"loading Amazon '{CATEGORY}' item text ...")
texts, item_ids = amazon_review.load_text(category=CATEGORY)

encoder = SentenceTransformer("sentence-t5-base", device=DEVICE)
features = encoder.encode(texts, batch_size=64, show_progress_bar=True)
del encoder  # release encoder memory
if DEVICE == "cuda":
    torch.cuda.empty_cache()

Let's sanity-check the embeddings before moving on: how many items, what's the embedding dimension, and what does an example item look like?

In [ ]:
print(f"number of items:     {len(item_ids)}")
print(f"feature matrix shape: {features.shape}")
print(f"\nexample item id:  {item_ids[0]}")
print(f"example item text: {texts[0][:200]}...")

Optionally, cache the embeddings to disk so you don't need to re-run the encoder later:

```python
np.savez("item_features.npz", features=features, item_ids=np.array(item_ids))
print(f"cached item content features -> item_features.npz  {features.shape}")
```

## Step 2 — Train RQ-KMeans

With item content embeddings in hand, we fit the residual K-means tokenizer. Each item will end up with a semantic ID of `NUM_LEVELS + 1` tokens: one code per residual level, plus the disambiguation token.

In [ ]:
NUM_LEVELS = 3        # number of semantic-ID levels (residual codebooks)
CODEBOOK_SIZE = 256   # number of codes (clusters) per level

rq_kmeans = RQKMeans(
    num_levels=NUM_LEVELS,
    codebook_size=CODEBOOK_SIZE,
    device=DEVICE,
    seed=123,
).fit(features)

## Step 3 — Inspect the resulting semantic-ID / token structure

Now let's look at what the tokenizer actually produced: the shape of the semantic ID table, a few example items, how many collisions needed disambiguation, and how evenly each codebook is used.

In [ ]:
sid_table = rq_kmeans.sid_table       # (n_items, num_levels + 1) int64
level_sizes = rq_kmeans.level_sizes   # e.g. [256, 256, 256, max_collisions+1]

print(f"items tokenized:        {sid_table.shape[0]}")
print(f"levels incl. dedup:     {len(level_sizes)}")
print(f"level sizes:            {level_sizes}")
print(f"semantic ID length:     {sid_table.shape[1]} tokens/item")

### Sample semantic IDs

A handful of example semantic IDs, mapped back to the original item ids and their source text.

In [ ]:
print("sample item -> semantic ID:")
for i in range(5):
    print(f"  {item_ids[i]}  ->  {tuple(int(v) for v in sid_table[i])} -> {texts[i]}")

### Collision stats

Items sharing the same `(level_0, ..., level_{N-1})` tuple collide on the same code — this is exactly why the disambiguation token exists.

In [ ]:
n_collisions = int((sid_table[:, -1] > 0).sum())
print(
    f"collisions: {n_collisions} / {sid_table.shape[0]} items "
    f"({100 * n_collisions / sid_table.shape[0]:.2f}%) shared a code tuple "
    f"and needed a disambiguation token (dedup level size = {level_sizes[-1]})"
)

### Codebook utilization per level

Are all clusters actually being used, or is usage skewed toward a handful of codes?

In [ ]:
print("codebook utilisation per level:")
for level in range(NUM_LEVELS):
    codes, counts = np.unique(sid_table[:, level], return_counts=True)
    print(
        f"  level {level}: {len(codes)}/{level_sizes[level]} codes used | "
        f"most-used code holds {counts.max()} items | "
        f"least-used holds {counts.min()} item(s) | "
        f"mean {counts.mean():.1f} items/code"
    )

### Codebook usage entropy

Raw utilization counts tell you *whether* every code is used, but not *how evenly*. The standard RQ-VAE / RQ-KMeans diagnostic for **codebook collapse** is Shannon entropy of the usage distribution: is usage spread across codes, or dominated by a few?

We compute:

- **Per-level entropy**: how uniformly each level's codes are used, normalized against the theoretical maximum (`log2(codebook_size)`), plus perplexity (`2^H`), the "effective" number of codes in use.
- **Joint entropy**: the same idea over the full `(level_0, ..., level_{N-1})` tuple, which tells you how much of the theoretical `codebook_size^num_levels` ID space is actually being used (the disambiguation column is excluded, since it's deterministic given a collision rather than a learned code).

In [ ]:
def shannon_entropy_bits(counts):
    """Shannon entropy in bits of the empirical distribution given by `counts`
    (an array of non-negative integer/float counts, need not sum to 1)."""
    counts = np.asarray(counts, dtype="float64")
    probs = counts[counts > 0] / counts.sum()
    return float(-(probs * np.log2(probs)).sum())


def codebook_entropy_report(sid_table, level_sizes, num_levels):
    """Per-level Shannon entropy of codebook usage (raw, normalized by the
    max-possible log2(codebook_size), and perplexity = 2**H), plus the joint
    entropy over the first `num_levels` columns (excludes the disambiguation
    level, which is deterministic given a collision and not a learned code).
    """
    report = {"per_level": []}
    for level in range(num_levels):
        _, counts = np.unique(sid_table[:, level], return_counts=True)
        h = shannon_entropy_bits(counts)
        h_max = np.log2(level_sizes[level])
        report["per_level"].append(
            {
                "level": level,
                "entropy_bits": h,
                "normalized_entropy": h / h_max,   # 1.0 = perfectly uniform usage
                "perplexity": 2**h,                # "effective" number of codes used
                "codebook_size": level_sizes[level],
            }
        )

    # joint entropy over the (level_0, ..., level_{num_levels-1}) tuples,
    # i.e. how much of the theoretical codebook_size**num_levels ID space is
    # actually used (dedup/collision column intentionally excluded)
    joint_codes = [tuple(row) for row in sid_table[:, :num_levels]]
    _, joint_counts = np.unique(joint_codes, axis=0, return_counts=True)
    h_joint = shannon_entropy_bits(joint_counts)
    h_joint_max = sum(np.log2(level_sizes[level]) for level in range(num_levels))
    report["joint"] = {
        "entropy_bits": h_joint,
        "normalized_entropy": h_joint / h_joint_max,
        "perplexity": 2**h_joint,               # effective # of distinct (pre-dedup) IDs used
        "max_possible_ids": int(np.prod(level_sizes[:num_levels])),
        "distinct_ids_used": len(joint_counts),
    }
    return report

In [ ]:
entropy_report = codebook_entropy_report(sid_table, level_sizes, NUM_LEVELS)

print("codebook usage entropy")
print("-" * 60)
for lvl in entropy_report["per_level"]:
    print(
        f"  level {lvl['level']}: H = {lvl['entropy_bits']:.2f} bits "
        f"(normalized {lvl['normalized_entropy']:.3f}, "
        f"perplexity {lvl['perplexity']:.1f} / {lvl['codebook_size']} codes)"
    )

j = entropy_report["joint"]
print(
    f"  joint (all {NUM_LEVELS} levels): H = {j['entropy_bits']:.2f} bits "
    f"(normalized {j['normalized_entropy']:.3f}, perplexity {j['perplexity']:.1f}), "
    f"{j['distinct_ids_used']}/{j['max_possible_ids']} possible pre-dedup IDs used"
)

A normalized entropy close to `1.0` at every level means usage is close to uniform across codes — a healthy sign that the tokenizer isn't collapsing onto a small subset of codes. Lower values point to codebook collapse: a few codes absorbing most of the items while the rest sit nearly empty.

Optionally, save the final tokenizer (with entropy report)

```python
with open("rq_kmeans_tokenizer.pkl", "wb") as f:
    pickle.dump(
        {
            "sid_table": sid_table,
            "level_sizes": level_sizes,
            "item_ids": list(item_ids),
            "centroids": rq_kmeans.centroids,  # per-level (K, feat_dim) centroid arrays
            "entropy_report": entropy_report,
        },
        f,
    )
print("saved tokenizer + semantic IDs -> rq_kmeans_tokenizer.pkl")
```